# 04 — Logging

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- remplacer vos `print` par des logs structurés ;
- utiliser les cinq niveaux `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL` ;
- configurer le logging minimal avec `logging.basicConfig` ;
- obtenir un logger par module.

## Prérequis

- exceptions, fonctions ;
- modules standard.

Pas encore vus :

- CLI argparse (notebook suivant), pytest.

## Plan

1. Pourquoi `logging` plutôt que `print` ?
2. Les 5 niveaux
3. `basicConfig` minimal
4. Format et date
5. Un logger par module
6. Logger une exception
7. Niveaux en fonction de l'environnement
8. Synthèse
9. Exercices

---


## 1. Pourquoi `logging` plutôt que `print` ?

- **Niveaux** : on peut filtrer au runtime (DEBUG en dev, WARNING en prod).
- **Destination** : console, fichier, syslog, etc.
- **Format** : timestamp, niveau, nom de logger — uniformes partout.
- **Activable par module** : certains modules en DEBUG, d'autres en INFO.
- **Zéro coût** quand le niveau est désactivé (test rapide sur le level).

---


## 2. Les 5 niveaux

| Niveau | Quand |
|---|---|
| `DEBUG` | Détails très fins, utiles pour le dev |
| `INFO` | Étapes normales du programme |
| `WARNING` | Quelque chose d'inattendu, mais on continue |
| `ERROR` | Une opération a échoué |
| `CRITICAL` | Le programme ne peut plus fonctionner |

Le **niveau actif** détermine les messages affichés. Par défaut : `WARNING`.

In [ ]:
import logging

logging.debug('niveau debug')
logging.info('niveau info')
logging.warning('niveau warning')
logging.error('niveau error')

Sans configuration, seules `warning` et `error` s'affichent.

---


## 3. `basicConfig` minimal

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

logging.info('démarrage')
logging.debug('non affiché — en dessous de INFO')
logging.warning('attention')

**`basicConfig` doit être appelé UNE SEULE FOIS** par programme, typiquement au tout début du point d'entrée.

---


## 4. Format et date

In [ ]:
import logging
import importlib
importlib.reload(logging)  # pour pouvoir reconfigurer dans le notebook

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)

logging.info('message formaté')

### Principaux placeholders

| Placeholder | Valeur |
|---|---|
| `%(asctime)s` | Horodatage |
| `%(levelname)s` | Niveau (`INFO`, …) |
| `%(name)s` | Nom du logger |
| `%(message)s` | Message |
| `%(module)s` | Nom de module |
| `%(funcName)s` | Nom de fonction |

---


## 5. Un logger par module

Chaque module crée son propre logger, nommé d'après `__name__`. Cela permet de filtrer au niveau de la hiérarchie.

In [ ]:
import logging

logger = logging.getLogger(__name__)

def demarrer() -> None:
    logger.info('démarrage')

demarrer()

---


## 6. Logger une exception

`logger.exception(msg)` capture automatiquement la traceback de l'exception courante. À utiliser **dans un bloc `except`**.

In [ ]:
import logging
logger = logging.getLogger(__name__)

def diviser(a: float, b: float) -> float:
    try:
        return a / b
    except ZeroDivisionError:
        logger.exception('division par zéro')
        return 0.0

diviser(10, 0)

---


## 7. Niveaux en fonction de l'environnement

Pattern fréquent : lire le niveau depuis une variable d'environnement.

In [ ]:
import os
import logging

niveau_str = os.environ.get('LOG_LEVEL', 'INFO')
niveau = getattr(logging, niveau_str.upper(), logging.INFO)
# logging.basicConfig(level=niveau, format='%(levelname)s: %(message)s')
print('niveau logging :', niveau_str)

---


## 8. Synthèse

| Besoin | Appel |
|---|---|
| Logger de module | `logger = logging.getLogger(__name__)` |
| Message d'info | `logger.info(msg)` |
| Avertissement | `logger.warning(msg)` |
| Erreur avec traceback | `logger.exception(msg)` |
| Config globale | `logging.basicConfig(level=..., format=...)` |

### Règles

1. **Jamais de `print`** dans du code de production.
2. Un `logger` par module, nommé `__name__`.
3. `logger.exception` dans les `except` — évite d'écrire `str(err)` à la main.

---


## 9. Exercices

### Exercice 1 — Convertir un print *(facile)*

Convertir ce script en utilisant `logging.info` :
```python
print('démarrage')
print('lecture config')
print('fini')
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Logging_basique", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

logging.info('démarrage')
logging.info('lecture config')
logging.info('fini')
```

</details>

### Exercice 2 — Logger d'un module *(facile)*

Écrire un module fictif `compte.py` qui obtient un logger via `logging.getLogger(__name__)` et écrit un message `info` dans une fonction `creer_compte(nom: str) -> None`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Logging_basique", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging
logger = logging.getLogger('compte')

def creer_compte(nom: str) -> None:
    """Crée un compte pour `nom`."""
    logger.info('création du compte %s', nom)

logging.basicConfig(level=logging.INFO, format='%(name)s %(levelname)s %(message)s')
creer_compte('Alice')
```

</details>

### Exercice 3 — `logger.exception` *(moyen)*

Écrire `lire_entier_sur(texte: str) -> int | None` qui tente `int(texte)` ; en cas d'échec, logguer l'exception avec `logger.exception` et renvoyer `None`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Logging_basique", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging
logger = logging.getLogger(__name__)

def lire_entier_sur(texte: str) -> int | None:
    """Renvoie int(texte) ou None en cas d'échec (en loggant)."""
    try:
        return int(texte)
    except ValueError:
        logger.exception('conversion échouée pour %r', texte)
        return None

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
lire_entier_sur('abc')
```

</details>

### Exercice 4 — Niveau depuis env *(moyen)*

Écrire une fonction `configurer(log_level_env: str = 'LOG_LEVEL') -> None` qui lit la variable d'environnement correspondante et appelle `basicConfig` avec ce niveau (par défaut `INFO`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Logging_basique", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging
import os

def configurer(log_level_env: str = 'LOG_LEVEL') -> None:
    """Configure le logging depuis une variable d'environnement."""
    niveau_str = os.environ.get(log_level_env, 'INFO').upper()
    niveau = getattr(logging, niveau_str, logging.INFO)
    logging.basicConfig(level=niveau, format='%(levelname)s %(message)s')

os.environ['LOG_LEVEL'] = 'WARNING'
configurer()
logging.info('non affiché')
logging.warning('affiché')
```

</details>

### Exercice 5 — Log vers fichier *(difficile)*

Écrire `configurer_fichier(chemin: str) -> None` qui configure le logging pour écrire en plus dans un fichier (en plus de la console). Astuce : utiliser `logging.FileHandler` et `getLogger().addHandler`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Logging_basique", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging

def configurer_fichier(chemin: str) -> None:
    """Ajoute un handler fichier au logger racine."""
    handler = logging.FileHandler(chemin, encoding='utf-8')
    handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.addHandler(handler)

configurer_fichier('/tmp/demo.log')
logging.info('hello fichier')
```

</details>

---


## Ressources externes

- [`logging` — doc](https://docs.python.org/3/library/logging.html)
- [Tutoriel logging (officiel)](https://docs.python.org/3/howto/logging.html)

---

## Mini-exemples supplémentaires

### Logger avec arguments interpolés (LAZY)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

nom = 'Alice'
age = 30

# ✅ bon — l'interpolation n'a lieu que si le log est effectivement émis
logging.info('utilisateur %s a %d ans', nom, age)

### Éviter l'interpolation eager

In [ ]:
# ❌ mauvais : la f-string est évaluée MÊME si le niveau est désactivé
# logging.debug(f'util {nom} age {age}')

# ✅ bon : évaluation paresseuse
logging.debug('util %s age %d', nom, age)

### Un logger par sous-système

In [ ]:
import logging

logger_db = logging.getLogger('app.db')
logger_http = logging.getLogger('app.http')

logger_db.info('connexion établie')
logger_http.info('GET /api/users')

### Filtrage par hiérarchie de noms

In [ ]:
import logging
logging.getLogger('app.db').setLevel(logging.WARNING)
logging.getLogger('app.http').setLevel(logging.DEBUG)
# Les loggers avec préfixe 'app.db' seront plus silencieux que ceux de 'app.http'
print('niveaux différents par sous-système')

### Format avec nom de fonction

In [ ]:
import logging
import importlib
importlib.reload(logging)

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s [%(funcName)s] %(message)s',
)

def process() -> None:
    logging.info('traitement en cours')

process()

### `logging.config.dictConfig`

In [ ]:
import logging.config

config = {
    'version': 1,
    'formatters': {
        'detail': {'format': '%(asctime)s %(name)s %(levelname)s %(message)s'},
    },
    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
            'formatter': 'detail',
        },
    },
    'root': {'level': 'INFO', 'handlers': ['console']},
}

# logging.config.dictConfig(config)
print('voir config dict')

Pour un vrai projet, on met la config dans un fichier (`logging.yaml`, `logging.json`) et on la charge au démarrage.

### Ne pas logger d'info sensible

Règle : **ne jamais logger** mots de passe, jetons, clés d'API, numéros de carte. Les logs sont souvent archivés et consultables.

---

## Quiz flash — vérifiez vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre compréhension avant de passer au notebook suivant. Les réponses sont dans le bloc `<details>` en dessous.


**Question 1.** Quels sont les 5 niveaux de log ?

<details>
<summary>📖 Réponse</summary>

`DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`.

</details>

**Question 2.** Niveau par défaut ?

<details>
<summary>📖 Réponse</summary>

`WARNING`.

</details>

**Question 3.** Comment logger une exception avec sa traceback ?

<details>
<summary>📖 Réponse</summary>

`logger.exception('message')` à l'intérieur d'un bloc `except`.

</details>

**Question 4.** Pourquoi `logger.info('nom %s', nom)` est-il meilleur que `logger.info(f'nom {nom}')` ?

<details>
<summary>📖 Réponse</summary>

L'interpolation paresseuse n'a lieu que si le log est réellement émis — économie en mode DEBUG désactivé.

</details>

**Question 5.** Comment obtenir un logger par module ?

<details>
<summary>📖 Réponse</summary>

`logger = logging.getLogger(__name__)`.

</details>

---

## Cheat sheet — `logging` quotidien

In [ ]:
# Pattern typique du point d'entrée
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)

In [ ]:
# Pattern dans chaque module
import logging
logger = logging.getLogger(__name__)

def traiter(valeurs: list[int]) -> int:
    logger.debug('traitement de %d valeurs', len(valeurs))
    resultat = sum(valeurs)
    logger.info('somme = %d', resultat)
    return resultat

traiter([1, 2, 3])

### Correspondance niveau → usage

| Niveau | Exemple |
|---|---|
| `DEBUG` | `'x = %s'` |
| `INFO` | `'démarrage du service'` |
| `WARNING` | `'cache vide, rebuild'` |
| `ERROR` | `'impossible d\'ouvrir la DB'` |
| `CRITICAL` | `'config invalide, arrêt'` |

### `print` vs `logging` — les 5 différences

1. **Niveaux** — filtrage sans modifier le code.
2. **Format uniforme** — timestamp, niveau, module.
3. **Destination configurable** — console, fichier, syslog, Sentry…
4. **Coût zéro** quand un niveau est désactivé.
5. **Thread-safe** dans les applications concurrentes.